In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

MODEL_NAME = "distilbert/distilbert-base-uncased"

imdb = load_dataset("imdb")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

sample_review = imdb["train"][0]
tokenized_sample = tokenizer(sample_review["text"], truncation=True)

{
    "label": sample_review["label"],
    "text_preview": sample_review["text"][:300],
    "token_count": len(tokenized_sample["input_ids"]),
}

/Users/alieladi/Dev/Chanterelles/chanterelle-examples/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating unsupervised split: 100%|██████████| 50000/50000 [00:00<00:00, 1316595.31 examples/s]


{'label': 0,
 'text_preview': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really h',
 'token_count': 363}

In [2]:
import torch
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

review_text = "This movie was surprisingly good. The acting carried the whole story."
inputs = tokenizer(review_text, return_tensors="pt", truncation=True)
with torch.no_grad():
    logits = model(**inputs).logits
predicted_class_id = logits.argmax().item()
confidence = torch.softmax(logits, dim=1)[0, predicted_class_id].item()

{
    "predicted_class_id": predicted_class_id,
    "confidence": round(confidence, 4),
}

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 14856.56it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'predicted_class_id': 1, 'confidence': 0.5109}